In [1]:
cd ..

/Users/nguyenhuyhai/20232/Centic/RecSys/reputation-analysis


In [136]:
wallet1 = {
  "_id": "0x1_0x0efddcf523ffe33a6f6afa146563b3e3ae070932",
  "wallet": "0x008a74d96d799b0fcfae8462bfff8c37c7ccc611",
  "interactedWallet": {
    "0x74de5d4fcbf63e00296fd95d33236b9794016631": {
      "contractAddress": {
        "0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48": {
          "timestampHistory": [
            121,
          ],
          "valueInUSDHistory": [
            121,
          ],
          "amountHistory": [
            121,
          ]
        },
        "ahihihihi": {
          "timestampHistory": [
            121,
          ],
          "valueInUSDHistory": [
            121,
          ],
          "amountHistory": [
            121,
          ]
        }
      }
    },
    "ahihi": {
      "contractAddress": {
        "0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48": {
          "timestampHistory": [
            121,
          ],
          "valueInUSDHistory": [
            121,
          ],
          "amountHistory": [
            121,
          ]
        }
      }
    }
  },
  "chainId": "0x1",
  "dailyAllTransactions": {

    "1212":400
  },
  "type": "wallet",
  "numTransaction": 36,
  "lattestTimestamp": 1715419019,
  "tags": {
    "1212": [
      "pepe_holders",
      "eth_holders",
      "nft_users"
    ]
  }
}





wallet2 = {
  "_id": "0x1_0x017d534264e4580bde76e84bbdc2b2c3bd477605",
  "wallet": "0x017d534264e4580bde76e84bbdc2b2c3bd477605",
  "interactedWallet": {
    "0x99ec373bb76d40ad6f456b848094dd16dd8460bc": {
      "contractAddress": {
        "0x0000000000000000000000000000000000000000": {
          "timestampHistory": [
            1715419019, 111
          ],
          "valueInUSDHistory": [
            21.5932918876,114
          ],
          "amountHistory": [
            0.00741803, 115
          ]
        }
      }
    }
  },
  "chainId": "0x1",
  "dailyAllTransactions": {
    "1680566400": 2,
    "1212":121
  },
  "type": "wallet",
  "numTransaction": 36,
  "lattestTimestamp": 1715419019,
  "tags": {
    "1212": [
      "pepe_holders",
      "eth_holders",
      "nft_users"
    ]
  }
}

In [137]:
wallets =[]
wallets.append(wallet1)
wallets.append(wallet2)

In [139]:
from databases.mongodb import MongoDB
from databases.mongodb_entity import MongoDBEntity
from databases.postgresql import PostgresDB
klg = MongoDBEntity()
mongodb = MongoDB()

mongodb.update_docs(collection_name='sos', data =  wallets, merge=True, flatten=False)

In [5]:
from utils.generate_training_utils import *

In [6]:
klg_cursor = klg._db.wallets.find({'_id':'0xfa_0x19246d4cb92671d0f2166d00c9de687ead3206f8'},{'_id':1,'balanceInUSD':1,'balanceChangeLogs':1})
dapps_balance = list(klg_cursor)[0]


In [9]:
import pandas as pd
pd.DataFrame([dapps_balance])
balance_df = search_balance_feature(
                        dapps_balance=dapps_balance, 
                        timestamp=1716366512
                        )

In [10]:
balance_df

,_id,overallBalance
0,0xfa_0x19246d4cb92671d0f2166d00c9de687ead3206f8,8727.708929


In [13]:
data = {'_id': '0x1_0xb523b34e21f240487144f24a38160cb4bbe596c4', 'interactedWallet': {'0xd0339bf46291cc45f8e1e70e502034cc99b9fea5': {'contractAddress': {'0x08d967bb0134f2d07f7cfb6e246680c53927dd30': {'timestampHistory': [1712952875, 1712623523, 1713017375, 1713042371], 'valueInUSDHistory': [3195.3372008226866, 5758.772648173631, 3160.430517718061, 18962.583106308364], 'amountHistory': [5000.0, 9868.8948, 30000.0]}}}}}


def get_df(data):
    rows = []
    for wallet, wallet_data in data['interactedWallet'].items():
        for contract, contract_data in wallet_data['contractAddress'].items():
            for i in range(len(contract_data['timestampHistory'])):
                print(i)
                row = {
                    '_id': data['_id'],
                    'interactedWallet': wallet,
                    'contractAddress': contract,
                    'timestampHistory': contract_data['timestampHistory'][i],
                    'valueInUSDHistory': contract_data['valueInUSDHistory'][i],
                    'amountHistory': contract_data['amountHistory'][i]
                }
                rows.append(row)

    df = pd.DataFrame(rows)
    return df

get_df(data)

0
1
2
3


IndexError: list index out of range

In [39]:

from databases.mongodb import MongoDB
from databases.mongodb_entity import MongoDBEntity
from databases.postgresql import PostgresDB
klg = MongoDBEntity()
mongodb = MongoDB()

#mongodb.update_docs(collection_name='testing', data =  wallets, merge=True)

from services.transaction_info_service import GetTransactionInfoService
from utils.list_dict_utils import get_special_wallet_list


In [18]:
special_wallets = get_special_wallet_list(hot_path='/Users/nguyenhuyhai/20232/Centic/RecSys/reputation-analysis/data/special_wallets/hot_wallets.json',
                                          burn_mint_path='/Users/nguyenhuyhai/20232/Centic/RecSys/reputation-analysis/data/special_wallets/burn_wallets.json')


[05-22-2024 16:02:20 +07] [INFO] [Getting special wallets jobs] - Done!!!!
[05-22-2024 16:02:20 +07] [INFO] [Getting special wallets jobs] - Done!!!!


In [55]:
def _wallet_transfer_info(directions, special_wallets, postgre, start_block, end_block, historical_token_price, block_timestamp_mappings, works): 
        for direction in directions:
            print(direction)
            if direction == 'from':
                opposite_direction ='to'
                wallet_transfer, failed_sender = postgre.query_addresses(
                    addresses=works, 
                    special_addresses=special_wallets, 
                    direction= direction, 
                    opposite_direction=opposite_direction, 
                    from_block=start_block, 
                    to_block=end_block, 
                    block_timestamp_mappings=block_timestamp_mappings, 
                    price_history = historical_token_price
                    )
                elite_sender = list(set(works)- set(failed_sender))
                if len(elite_sender) ==0:
                    return  pd.DataFrame(), pd.DataFrame(), []
                else:
                    if len(wallet_transfer)!=0:            
                        transfer_concat = pd.DataFrame(wallet_transfer)
                        transfer_concat = transfer_concat.dropna()
                        transfer_concat = transfer_concat.rename(
                            columns={"value":"amount", 
                                     "contract_address":"contractAddress"}
                                     )
                        wallet_transfer_info = transfer_concat.groupby(
                            ['from_address', 'to_address', 'contractAddress'])  \
                                .agg({"block_number":list, 
                                      'amount':list, 
                                      "valueInUSD": ['sum',list]}
                                    )
                        wallet_transfer_info.columns = ['{}_{}'.format(col[0], col[1]) for col in wallet_transfer_info.columns]
                        sending_transfer_info = wallet_transfer_info.reset_index().rename(
                                                columns={"from_address": "wallet",
                                                         "to_address":"interactedWallet", 
                                                         "block_number_list":"timestampHistory", 
                                                         "valueInUSD_list":"valueInUSDHistory",
                                                         "valueInUSD_sum":"totalValueInUSD", 
                                                         "amount_list":"amountHistory"})
                    else:
                        sending_transfer_info = pd.DataFrame()

            if direction =='to':
                opposite_direction = 'from'
                wallet_transfer, failed_borrower = postgre.query_addresses(
                    addresses=elite_sender, 
                    special_addresses=special_wallets, 
                    direction= direction, 
                    opposite_direction=opposite_direction, 
                    from_block=start_block, 
                    to_block=end_block, 
                    block_timestamp_mappings=block_timestamp_mappings, 
                    price_history = historical_token_price
                    )
                elite_borrower = list(set(elite_sender)- set(failed_borrower))
                if len(elite_borrower) ==0:
                    return  pd.DataFrame(), pd.DataFrame(), []
                else:
                    if len(wallet_transfer)!=0:            
                        transfer_concat = pd.DataFrame(wallet_transfer)
                        transfer_concat = transfer_concat.dropna()
                        transfer_concat = transfer_concat.rename(columns={"value":"amount", 
                                                                          "contract_address":"contractAddress"})
                        
                        wallet_transfer_info = transfer_concat.groupby(['from_address', 'to_address', 'contractAddress']).\
                                                                agg({"block_number":list, 
                                                                'amount':list, "valueInUSD": ['sum',list]})
                        wallet_transfer_info.columns = ['{}_{}'.format(col[0], col[1]) for col in wallet_transfer_info.columns]
                        receiving_transfer_info = wallet_transfer_info.reset_index()\
                                        .rename(columns={"to_address": "wallet",
                                                         "from_address":"interactedWallet", 
                                                         "block_number_list":"timestampHistory",
                                                         "valueInUSD_list":"valueInUSDHistory",            
                                                         "valueInUSD_sum":"totalValueInUSD", 
                                                         "amount_list":"amountHistory"})
                    else: 
                        receiving_transfer_info = pd.DataFrame()

        elite_addresses = list(set(elite_sender).intersection(set(elite_borrower)))
        if sending_transfer_info.empty == False:
            sending_transfer_info = sending_transfer_info[sending_transfer_info['wallet'].isin(elite_addresses)]
        if receiving_transfer_info.empty == False:
            receiving_transfer_info = receiving_transfer_info[receiving_transfer_info['wallet'].isin(elite_addresses)]

        return sending_transfer_info, receiving_transfer_info, elite_addresses

In [38]:
from databases.mongodb_main import MongoDBMain
from utils.history_price_utils import query_historical_prices


In [36]:
directions =['from', 'to']

In [29]:
historical_token_price = query_historical_prices(mongo_community=mongodb,
                                                     token_price='top_token_multichain',
                                                     chains=['0x1'])

[05-22-2024 16:17:47 +07] [INFO] [Query token historical price job] - Done!!!
[05-22-2024 16:17:47 +07] [INFO] [Query token historical price job] - Done!!!
[05-22-2024 16:17:47 +07] [INFO] [Query token historical price job] - Done!!!


In [115]:
mongo_transaction = MongoDBMain(chain='ethereum')

block_timestamp_mappings = mongo_transaction.get_block_number_to_timestamp(start_block=19485939, end_block=19823409)

In [56]:
df =_wallet_transfer_info(directions=directions,
                      special_wallets=special_wallets,
                      postgre=PostgresDB(chain_id='0x1'),
                      start_block=19485939,
                      end_block=19823409,
                      historical_token_price=historical_token_price['0x1'],
                      block_timestamp_mappings=block_timestamp_mappings,
                      works=['0xc0f310084e7c63c12e49e056a33a3176354a8e53']

                      )

df[0]

from
to


,wallet,interactedWallet,contractAddress,timestampHistory,amountHistory,totalValueInUSD,valueInUSDHistory
0,0xc0f310084e7c63c12e49e056a33a3176354a8e53,0x74de5d4fcbf63e00296fd95d33236b9794016631,0xebd9d99a3982d547c5bb4db7e3b1f9f14b67eb83,"[1713729767, 1713731843]","[16956.5, 20000.069509769444]",3655.596030,"[1677.269154, 1978.3268756283544]"
1,0xc0f310084e7c63c12e49e056a33a3176354a8e53,0xbcffa1619ab3ce350480ae0507408a3c6c3572bd,0xebd9d99a3982d547c5bb4db7e3b1f9f14b67eb83,"[1711471307, 1711473779, 1714496039]","[28334.55159102932, 15089.0, 26114.149]",14023.054596,"[7569.234575244133, 4030.8448199341947, 2422.9..."
2,0xc0f310084e7c63c12e49e056a33a3176354a8e53,0xe2d2ffc9513327b91cdf05ff536d72972ef90a3d,0xebd9d99a3982d547c5bb4db7e3b1f9f14b67eb83,[1711069487],[15089.212979044907],1540.168621,[1540.1686205657559]


In [57]:
df[0]

,wallet,interactedWallet,contractAddress,timestampHistory,amountHistory,totalValueInUSD,valueInUSDHistory
0,0xc0f310084e7c63c12e49e056a33a3176354a8e53,0x74de5d4fcbf63e00296fd95d33236b9794016631,0xebd9d99a3982d547c5bb4db7e3b1f9f14b67eb83,"[1713729767, 1713731843]","[16956.5, 20000.069509769444]",3655.596030,"[1677.269154, 1978.3268756283544]"
1,0xc0f310084e7c63c12e49e056a33a3176354a8e53,0xbcffa1619ab3ce350480ae0507408a3c6c3572bd,0xebd9d99a3982d547c5bb4db7e3b1f9f14b67eb83,"[1711471307, 1711473779, 1714496039]","[28334.55159102932, 15089.0, 26114.149]",14023.054596,"[7569.234575244133, 4030.8448199341947, 2422.9..."
2,0xc0f310084e7c63c12e49e056a33a3176354a8e53,0xe2d2ffc9513327b91cdf05ff536d72972ef90a3d,0xebd9d99a3982d547c5bb4db7e3b1f9f14b67eb83,[1711069487],[15089.212979044907],1540.168621,[1540.1686205657559]


In [60]:
df[1].iloc[0].amountHistory

[27856.625242069706, 28334.55159102932, 9099.944267699739, 15089.212979044907]

In [61]:
block_timestamp_mappings[19485940]

1711059371

In [64]:
#from services.get_exchange_info_service import GetTransactionInfoService

In [82]:
import pandas as pd 
import time 

from constants.network_constants import NATIVE_TOKEN
from utils.logger_utils import write_file
from utils.list_dict_utils import flatten_and_concat, rename_key, update_list_of_dict, chunk_list
from utils.history_price_utils import get_daily_timestamp
from utils.upload_utils import update_format
from databases.mongodb_main import MongoDBMain
from databases.postgresql import PostgresDB
from constants.network_constants import Chains

class GetTransactionInfoService():
    def __init__(self, mongo_community, historical_token_price, multichain_transaction, special_wallets, filename: str):
        self.profile_info = list()
        self.mongo_community = mongo_community
        self.historical_token_price = historical_token_price
        self.multichain_transaction = multichain_transaction
        self.special_wallets = special_wallets
        
        ## Get deposit, borrow, balance info
        self.direction = ['from','to']
        self.filename = filename
        self.native = NATIVE_TOKEN
        self.fail_addresses = list()

    def _generate_exchange_info(self, works):
        addresses = list()
        other_lst = list()
        for address in works:
            if (address['address'] not in self.special_wallets) &(address['type']!='contract'):     
                    address_dct = dict()
                    address_dct['chainId'] = address['chainId']
                    address_dct['address'] = address['address']
                    addresses.append(address_dct)
                    other_lst.append(address)
        
        multichain_address = update_list_of_dict(addresses)
    
        multichain_other_fields = update_list_of_dict(other_lst)
        for chain_id, addresses in multichain_address.items():
            if chain_id in self.multichain_transaction:
                chain = Chains.reverse_mapping[chain_id]
                other_info = rename_key(multichain_other_fields[chain_id], 'address','wallet') 
                sending_transfer_info, receiving_transfer_info, transfer_elite_addresses = self._wallet_transfer_info(
                    postgre=PostgresDB(chain_id=chain_id),
                    start_block=self.multichain_transaction[chain_id]['start_block'],
                    end_block=self.multichain_transaction[chain_id]['end_block'],
                    historical_token_price=self.historical_token_price[chain_id],
                    block_timestamp_mappings=self.multichain_transaction[chain_id]['block_timestamp_mappings'],
                    works=addresses
                    ) #postgres

                if len(transfer_elite_addresses)!=0:
                    sending_transaction_info, receiving_transaction_info = self._wallet_transaction_info(
                        mongo_transaction=MongoDBMain(chain=chain),
                        start_date=self.multichain_transaction[chain_id]['start'],
                        end_date=self.multichain_transaction[chain_id]['end'],
                        historical_token_price=self.historical_token_price[chain_id],
                        works=addresses
                    ) #mongo
                    if sending_transaction_info.empty==False:
                        sending_transaction_info['contractAddress'] = self.native
                    if receiving_transaction_info.empty==False:    
                        receiving_transaction_info['contractAddress'] = self.native
                    
                    profiling = pd.concat([sending_transfer_info, sending_transaction_info, receiving_transaction_info, receiving_transfer_info])
                    profiling = profiling.dropna()
                    if (profiling.empty == False) &('contractAddress' in profiling.columns):
                        try:
                            profile_info = profiling.groupby(
                                ['wallet','interactedWallet','contractAddress']) \
                                .agg({"timestampHistory":flatten_and_concat,
                                       "valueInUSDHistory":flatten_and_concat,
                                       "amountHistory":flatten_and_concat}
                                    ).reset_index()
                            
                            profile_timestamp = profile_info[['wallet','timestampHistory']].copy()
                            profile_max_timestamp = profile_timestamp.groupby('wallet').agg(
                                {"timestampHistory":flatten_and_concat}
                            )
                            profile_max_timestamp['lattestTimestamp'] = profile_max_timestamp['timestampHistory'].apply(lambda x: max(x))
                            max_timestamp = profile_max_timestamp['lattestTimestamp'].to_dict()
                            profile_info_dict = update_format(profile_info)
                            dict_lookup = {item['wallet']: item for item in profile_info_dict}
                            for update in other_info:
                                wallet = update['wallet']
                                if (wallet in dict_lookup):
                                    max_time_dct = dict()
                                    tags_dct = dict()
                                    tags_time = dict()
                                    max_time_dct['lattestTimestamp'] = max_timestamp[wallet]
                                    max_time_dct['lattestTags'] = update['tags']
                                    now_time = time.time()
                                    now_daily_time = get_daily_timestamp(int(now_time))
                                    tags_time[f'{now_daily_time}'] = update['tags']
                                    tags_dct['tags'] = tags_time
                                    dict_lookup[wallet].update(update)  # Update in here
                                    dict_lookup[wallet].update(max_time_dct) 
                                    dict_lookup[wallet].update(tags_dct)

                            update_jobs = chunk_list(input_list=profile_info_dict, chunk_size=200)
                            return update_jobs
                            for job in update_jobs:
                     #           self.mongo_community.update_docs(collection_name=self.filename, data=job, merge=True)
                                print(job)
                        except Exception as e:
                            profiling.to_csv(f"data/log_file/address_info/{chain_id}/checking.csv", index=False)
                            print('oh no having error in combinning 2 dataframe')
                            print(e)
                else:
                    self.fail_addresses.extend(addresses)
     

    def _wallet_transaction_info(self, mongo_transaction, start_date, end_date, historical_token_price, works): 
        for direction in self.direction:
            if direction == 'from':
                opposite_direction ='to'
                address_info_lst = mongo_transaction.get_transaction_info(
                    start_date=start_date, 
                    end_date=end_date, 
                    direction=direction, 
                    opposite_direction=opposite_direction, 
                    special_addresses=self.special_wallets, 
                    addresses=works, 
                    price_history=historical_token_price[f'{self.native}']
                    )

                if len(address_info_lst) !=0:
                    addresses_info_df = pd.DataFrame(address_info_lst)
                    wallet_transaction_info = addresses_info_df.groupby(['from_address', 'to_address'])\
                                                                        .agg({"block_timestamp":list,  
                                                                            'amount':list, 
                                                                            "valueInUSD": ['sum', list]})
                    
                    wallet_transaction_info.columns = ['{}_{}'.format(col[0], col[1]) for col in wallet_transaction_info.columns]
                    sending_transaction_info = wallet_transaction_info.reset_index()\
                                        .rename(
                                            columns={"from_address": "wallet",
                                                     "to_address":"interactedWallet",
                                                     "block_timestamp_list":"timestampHistory",
                                                     "valueInUSD_list":"valueInUSDHistory", 
                                                     "valueInUSD_sum":"totalValueInUSD", 
                                                     "amount_list":"amountHistory"})
                
                if (len(address_info_lst) ==0 ):
                    sending_transaction_info = pd.DataFrame()

            if direction == 'to':
                opposite_direction ='from'
                address_info_lst = mongo_transaction.get_transaction_info(
                    start_date=start_date, 
                    end_date=end_date, 
                    direction=direction, 
                    opposite_direction=opposite_direction, 
                    special_addresses=self.special_wallets, 
                    addresses=works, 
                    price_history=historical_token_price[f'{self.native}']
                    )

                if len(address_info_lst) !=0:
                    addresses_info_df = pd.DataFrame(address_info_lst)
                    wallet_transaction_info = addresses_info_df.groupby(['from_address', 'to_address'])\
                                                                        .agg({"block_timestamp":list,  
                                                                              'amount':list, 
                                                                              "valueInUSD": ['sum', list]})
                    
                    wallet_transaction_info.columns = ['{}_{}'.format(col[0], col[1]) for col in wallet_transaction_info.columns]
                    receiving_transaction_info = wallet_transaction_info.reset_index()\
                                        .rename(
                                            columns={"to_address": "wallet",
                                                     "from_address":"interactedWallet", 
                                                     "block_timestamp_list":"timestampHistory", 
                                                     "valueInUSD_list":"valueInUSDHistory", 
                                                     "valueInUSD_sum":"totalValueInUSD", 
                                                     "amount_list":"amountHistory"})
                    
                if (len(address_info_lst) ==0 ):
                    receiving_transaction_info = pd.DataFrame()

        return sending_transaction_info, receiving_transaction_info


    def _wallet_transfer_info(self, postgre, start_block, end_block, historical_token_price, block_timestamp_mappings, works): 
        for direction in self.direction:
            if direction == 'from':
                opposite_direction ='to'
                wallet_transfer, failed_sender = postgre.query_addresses(
                    addresses=works, 
                    special_addresses=self.special_wallets, 
                    direction= direction, 
                    opposite_direction=opposite_direction, 
                    from_block=start_block, 
                    to_block=end_block, 
                    block_timestamp_mappings=block_timestamp_mappings, 
                    price_history = historical_token_price
                    )
                elite_sender = list(set(works)- set(failed_sender))
                if len(elite_sender) ==0:
                    return  pd.DataFrame(), pd.DataFrame(), []
                else:
                    if len(wallet_transfer)!=0:            
                        transfer_concat = pd.DataFrame(wallet_transfer)
                        transfer_concat = transfer_concat.dropna()
                        transfer_concat = transfer_concat.rename(
                            columns={"value":"amount", 
                                     "contract_address":"contractAddress"}
                                     )
                        wallet_transfer_info = transfer_concat.groupby(
                            ['from_address', 'to_address', 'contractAddress'])  \
                                .agg({"block_number":list, 
                                      'amount':list, 
                                      "valueInUSD": ['sum',list]}
                                    )
                        wallet_transfer_info.columns = ['{}_{}'.format(col[0], col[1]) for col in wallet_transfer_info.columns]
                        sending_transfer_info = wallet_transfer_info.reset_index().rename(
                                                columns={"from_address": "wallet",
                                                         "to_address":"interactedWallet", 
                                                         "block_number_list":"timestampHistory", 
                                                         "valueInUSD_list":"valueInUSDHistory",
                                                         "valueInUSD_sum":"totalValueInUSD", 
                                                         "amount_list":"amountHistory"})
                    else:
                        sending_transfer_info = pd.DataFrame()

            if direction =='to':
                opposite_direction = 'from'
                wallet_transfer, failed_borrower = postgre.query_addresses(
                    addresses=elite_sender, 
                    special_addresses=self.special_wallets, 
                    direction= direction, 
                    opposite_direction=opposite_direction, 
                    from_block=start_block, 
                    to_block=end_block, 
                    block_timestamp_mappings=block_timestamp_mappings, 
                    price_history = historical_token_price
                    )
                elite_borrower = list(set(elite_sender)- set(failed_borrower))
                if len(elite_borrower) ==0:
                    return  pd.DataFrame(), pd.DataFrame(), []
                else:
                    if len(wallet_transfer)!=0:            
                        transfer_concat = pd.DataFrame(wallet_transfer)
                        transfer_concat = transfer_concat.dropna()
                        transfer_concat = transfer_concat.rename(columns={"value":"amount", 
                                                                          "contract_address":"contractAddress"})
                        
                        wallet_transfer_info = transfer_concat.groupby(['from_address', 'to_address', 'contractAddress']).\
                                                                agg({"block_number":list, 
                                                                'amount':list, "valueInUSD": ['sum',list]})
                        wallet_transfer_info.columns = ['{}_{}'.format(col[0], col[1]) for col in wallet_transfer_info.columns]
                        receiving_transfer_info = wallet_transfer_info.reset_index()\
                                        .rename(columns={"to_address": "wallet",
                                                         "from_address":"interactedWallet", 
                                                         "block_number_list":"timestampHistory",
                                                         "valueInUSD_list":"valueInUSDHistory",            
                                                         "valueInUSD_sum":"totalValueInUSD", 
                                                         "amount_list":"amountHistory"})
                    else: 
                        receiving_transfer_info = pd.DataFrame()

        elite_addresses = list(set(elite_sender).intersection(set(elite_borrower)))
        if sending_transfer_info.empty == False:
            sending_transfer_info = sending_transfer_info[sending_transfer_info['wallet'].isin(elite_addresses)]
        if receiving_transfer_info.empty == False:
            receiving_transfer_info = receiving_transfer_info[receiving_transfer_info['wallet'].isin(elite_addresses)]

        return sending_transfer_info, receiving_transfer_info, elite_addresses
    def _end(self):
        super()._end()
        write_file(f'address_info/', 'fail_addresses.json',self.fail_addresses)



In [253]:
from config import MongoDBConfig
from pymongo import MongoClient, UpdateOne, InsertOne
from utils.logger_utils  import get_logger
from utils.list_dict_utils import flatten_dict, delete_none

logger = get_logger("Mongo Community")

class MongoDB:
    def __init__(self, connection_url=None, chain_id: str = ""):
        if not connection_url:
            connection_url = MongoDBConfig.CONNECTION_URL
            
        self.connection_url = connection_url.split('@')[-1]
        self.connection = MongoClient(connection_url)
        self._db = self.connection[MongoDBConfig.DATABASE]


    def count_documents(self, col_name):
        return self._db[col_name].estimated_document_count()

    def get_documents(self, col_name: str, skip: int, limit: int):
        return self._db[col_name].find({}).skip(skip).limit(limit)

    def get_documents_by_ids(self, col_name: str, ids: list):
        return self._db[col_name].find({'_id': {'$in': ids}})


    def update_data(self, name, data):
        bulk_updates = [
            UpdateOne({'_id': datum['_id']}, {'$set': datum}, upsert=True
            )
            for datum in data 
       ]
        self._db[f"{name}"].bulk_write(bulk_updates)


    @staticmethod
    def create_update_doc(document, keep_none=False, merge=True, shard_key=None):
        unset, set_, add_to_set = [], [], []
        if not keep_none:
            doc = flatten_dict(document)
            for key, value in doc.items():
                if value is None:
                    tmp = {
                        "_id": document["_id"],
                        key: ""
                    }
                    if shard_key:
                        tmp[shard_key] = document[shard_key]
                    unset.append(tmp)
                    continue
                if not merge:
                    continue
                if (isinstance(value, list)&(key!='lattestTags')):
                    tmp = {
                        "_id": document["_id"],
                        key: {"$each": [i for i in value if i]}
                    }
                    if shard_key:
                        tmp[shard_key] = document[shard_key]
                    add_to_set.append(tmp)
                else:
                    tmp = {
                        "_id": document["_id"],
                        key: value
                    }
                    if shard_key:
                        tmp[shard_key] = document[shard_key]
                    set_.append(tmp)

        if not merge:
            if keep_none:
                set_.append(document)
            else:
                set_.append(delete_none(document))

        return unset, set_, add_to_set


    def update_docs(self, collection_name, data, keep_none=False, merge=True, shard_key=None, flatten=True):
        """If merge is set to True => sub-dictionaries are merged instead of overwritten"""
        col = self._db[collection_name]
        # col.insert_many(data, overwrite=True, overwrite_mode='update', keep_none=keep_none, merge=merge)
        bulk_operations = []
        if not flatten:
            if not shard_key:
                bulk_operations = [UpdateOne({"_id": item["_id"]}, {"$set": item}, upsert=True) for item in data]
            else:
                bulk_operations = [UpdateOne({"_id": item["_id"], shard_key: item[shard_key]}, {"$set": item}, upsert=True) for item in data]
            col.bulk_write(bulk_operations)
            return
        
        for document in data:
            unset, set_, add_to_set = self.create_update_doc(document, keep_none, merge, shard_key)
            print(unset)
            print(set_)
            print(add_to_set)
            if not shard_key:
                bulk_operations += [
                    UpdateOne({"_id": item["_id"]},
                                {"$unset": {key: value for key, value in item.items() if key != "_id"}}, upsert=True)
                    for item in unset]
                bulk_operations += [
                    UpdateOne({"_id": item["_id"]},
                                {"$set": {key: value for key, value in item.items() if key != "_id"}}, upsert=True)
                    for item in set_]
                bulk_operations += [
                    UpdateOne({"_id": item["_id"]},
                                {"$push": {key: value for key, value in item.items() if (key not in ["_id","lattestTags"])}},
                                upsert=True)
                    for item in add_to_set]
                # bulk_operations+= [
                #     UpdateOne({"_id": item["_id"]},
                #                  {"$set":{key: value for key, value in item.items() if key=="lattestTags"}},
                #                 upsert=True)
                #     for item in add_to_set]

            if shard_key:
                keys = ["_id", shard_key]
                bulk_operations += [
                    UpdateOne({"_id": item["_id"], shard_key: item[shard_key]},
                                {"$unset": {key: value for key, value in item.items() if key not in keys}},
                                upsert=True)
                    for item in unset]
                bulk_operations += [
                    UpdateOne({"_id": item["_id"], shard_key: item[shard_key]},
                                {"$set": {key: value for key, value in item.items() if key not in keys}}, upsert=True)
                    for item in set_]
                bulk_operations += [
                    UpdateOne({"_id": item["_id"], shard_key: item[shard_key]},
                                {"$push": {key: value for key, value in item.items() if key not in keys}},
                                upsert=True)
                    for item in add_to_set]
        col.bulk_write(bulk_operations)
    

  


In [260]:
wallet1 = {
  "_id": "0x1_0x0efddcf523ffe33a6f6afa146563b3e3ae070932",
  "wallet": "0x008a74d96d799b0fcfae8462bfff8c37c7ccc611",
  "interactedWallet": {
    "0x74de5d4fcbf63e00296fd95d33236b9794016631": {
      "contractAddress": {
        "0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48": {
          "timestampHistory": [
            112, 124
          ],
          "valueInUSDHistory": [
            128, 129
          ],
          "amountHistory": [
            130, 131
          ]
        },
        "ahihihihi": {
          "timestampHistory": [
            121,
          ],
          "valueInUSDHistory": [
            121,
          ],
          "amountHistory": [
            121,
          ]
        }
      }
    },
    "hihi": {
      "contractAddress": {
        "0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48": {
          "timestampHistory": [
            121,
          ],
          "valueInUSDHistory": [
            121,
          ],
          "amountHistory": [
            121,
          ]
        }
      }
    }
  },
  "chainId": "0x1",
  "dailyAllTransactions": {

    "1212":400
  },
  "type": "wallet",
  "numTransaction": 36,
  "lattestTimestamp": 1999,
  "lattestTags": ['a', 'b'],
  "tags": {
    "1212": [
      "pepe_holders",
      "eth_holders",
      "nft_users"
    ]
  }
}

In [261]:
mongodb = MongoDB()

In [262]:
mongodb.update_docs(collection_name='sos', data=[wallet1], merge=True)

[]
[{'_id': '0x1_0x0efddcf523ffe33a6f6afa146563b3e3ae070932'}, {'_id': '0x1_0x0efddcf523ffe33a6f6afa146563b3e3ae070932', 'wallet': '0x008a74d96d799b0fcfae8462bfff8c37c7ccc611'}, {'_id': '0x1_0x0efddcf523ffe33a6f6afa146563b3e3ae070932', 'chainId': '0x1'}, {'_id': '0x1_0x0efddcf523ffe33a6f6afa146563b3e3ae070932', 'dailyAllTransactions.1212': 400}, {'_id': '0x1_0x0efddcf523ffe33a6f6afa146563b3e3ae070932', 'type': 'wallet'}, {'_id': '0x1_0x0efddcf523ffe33a6f6afa146563b3e3ae070932', 'numTransaction': 36}, {'_id': '0x1_0x0efddcf523ffe33a6f6afa146563b3e3ae070932', 'lattestTimestamp': 1999}, {'_id': '0x1_0x0efddcf523ffe33a6f6afa146563b3e3ae070932', 'lattestTags': ['a', 'b']}]
[{'_id': '0x1_0x0efddcf523ffe33a6f6afa146563b3e3ae070932', 'interactedWallet.0x74de5d4fcbf63e00296fd95d33236b9794016631.contractAddress.0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48.timestampHistory': {'$each': [112, 124]}}, {'_id': '0x1_0x0efddcf523ffe33a6f6afa146563b3e3ae070932', 'interactedWallet.0x74de5d4fcbf63e00296fd95

In [221]:
multichain_transaction = dict()
small_dict = dict()
small_dict['start'] = 1711059351
small_dict['end'] = 1715147927
small_dict['start_block'] = 19485939
small_dict['end_block'] = 19823409
small_dict['block_timestamp_mappings'] =block_timestamp_mappings

multichain_transaction['0x1'] = small_dict

In [179]:
transac = GetTransactionInfoService(mongo_community=mongodb,
                                    historical_token_price=historical_token_price,
                                    multichain_transaction=multichain_transaction,
                                    special_wallets=special_wallets,
                                    filename='sos')

In [132]:
addresss = {'address':'0x0efddcf523ffe33a6f6afa146563b3e3ae070932', '_id':'0x1_0x0efddcf523ffe33a6f6afa146563b3e3ae070932','chainId':'0x1', 'type':'wallet', 'tags':['soss']}

In [133]:
info = transac._generate_exchange_info([addresss])

In [ ]:
info[0][0]

In [145]:
mongodb.update_docs(collection_name='sos', data=[wallet1], merge=True, flatten=False)